In [1]:
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
if os.environ['GEMINI_API_KEY']:
    print("API Key is set.")

API Key is set.


In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI

e:\Learning-Path\AI\RAG\RAG1\.venv\lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [5]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [6]:
response = llm.invoke("What is AI? Tell me in one line")
response.content[0]["text"]

'AI is the simulation of human intelligence by machines, enabling them to think, learn, and solve problems like humans.'

## **RAG Implementation with your own Text Data**

### **Step 1 : Preparing Document for your own Text**

In [7]:
from langchain_core.documents import Document

In [8]:
my_txt = """
"AI" redirects here. For other uses, see AI (disambiguation) and Artificial intelligence (disambiguation).

Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).

The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers use techniques including state space search and mathematical optimisation, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind, and Meta, aim to create artificial general intelligence (AGI)—AI that can complete nearly any cognitive task at least as well as a human.[3]

Artificial intelligence was founded as an academic discipline in 1956.[4] The field went through multiple cycles of optimism throughout its history,[5][6] followed by periods of disappointment and loss of funding, known as AI winters.[7][8] Funding and interest increased substantially after 2012, when graphics processing units (GPUs) started being used to accelerate neural networks, and deep learning outperformed previous AI techniques.[9] This growth accelerated further after 2017 with the transformer architecture.[10] In the 2020s, an AI boom coincided with advances in generative AI, which became widespread and allowed for the creation and modification of media. In addition to AI safety and unintended consequences and harms from the use of AI, ethical concerns, AI's long-term effects, environmental effects,[11] and potential existential risks have prompted discussions of AI regulation.
"""

In [9]:
docs = [Document(page_content=my_txt, metadata={"source":"ABC", "docID":"Doc1"})]
docs

[Document(metadata={'source': 'ABC', 'docID': 'Doc1'}, page_content='\n"AI" redirects here. For other uses, see AI (disambiguation) and Artificial intelligence (disambiguation).\n\nArtificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]\n\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g., text, images, audio, and videos).\n\nThe traditional goals of AI research include learning, reasoning, knowledg

### **Step 2 : Splitting the Document into Chunks**

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [11]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

In [12]:
chunks = splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'ABC', 'docID': 'Doc1'}, page_content='"AI" redirects here. For other uses, see AI (disambiguation) and Artificial intelligence (disambiguation).'),
 Document(metadata={'source': 'ABC', 'docID': 'Doc1'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]'),
 Document(metadata={'source': 'ABC', 'docID': 'Doc1'}, page_content='High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and c

In [13]:
len(chunks)

7

### **Step 3 : Creating Embeddings for the Chunks**

In [14]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [15]:
embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

### **Step 4 : Create and Store Embeddings in the Vector Store**

In [16]:
from langchain_community.vectorstores import Chroma

C:\Users\abhay\AppData\Local\Temp\ipykernel_8992\3485177327.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [17]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

### **Step 5 : Semantic Search**

In [18]:
vector_store.similarity_search("What is AI?", k=3)

[Document(metadata={'docID': 'Doc1', 'source': 'ABC'}, page_content='Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.[1]'),
 Document(metadata={'docID': 'Doc1', 'source': 'ABC'}, page_content='"AI" redirects here. For other uses, see AI (disambiguation) and Artificial intelligence (disambiguation).'),
 Document(metadata={'docID': 'Doc1', 'source': 'ABC'}, page_content='philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind, and Meta, aim to create artificial general intelligence (AGI)—AI that can complete near

## **Talk to LLM**

In [19]:
from langchain_google_genai import ChatGoogleGenerativeAI
from IPython.display import display, Markdown

In [20]:
context = vector_store.similarity_search("What is AI?", k=3)

In [21]:
response = llm.invoke(f"What is AI? You can answer using the following context : {context}")
display(Markdown(response.text))

Based on the provided context, Artificial Intelligence (AI) is defined as:

* The capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making.
* A field of research in engineering, mathematics, computer science, philosophy, neuroscience, and other fields that develops and studies methods and software. This technology enables machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.